In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import backend as K
from tensorflow.keras import layers, models, callbacks, optimizers
from sklearn.preprocessing import OneHotEncoder
import joblib
from pathlib import Path

from sklearn.metrics import classification_report, accuracy_score, roc_curve, auc, precision_recall_curve

from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, classification_report, brier_score_loss, confusion_matrix, log_loss
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight

In [28]:
"""
# Based on:
#   L. Delong, A. Kozak, "The use of autoencoders for training neural networks with
#   mixed categorical and numerical features"
#   Available: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3952470

"""

def max_loss_function(y_true, y_pred):
    return K.mean(y_true - y_pred)


def min_max_scaler(X: np.ndarray) -> np.ndarray:

    X = np.asarray(X, dtype=np.float32)
    if X.ndim == 1:
        x_min = X.min()
        x_max = X.max()
        if x_max == x_min:
            return np.zeros_like(X)
        return 2.0 * (X - x_min) / (x_max - x_min) - 1.0

    elif X.ndim == 2:
        mins = X.min(axis=0)
        maxs = X.max(axis=0)
        denom = (maxs - mins)
        denom[denom == 0.0] = 1.0
        return 2.0 * (X - mins) / denom - 1.0

    else:
        raise ValueError("min_max_scaler only supports 1D or 2D arrays.")


def per_feature_hidden_dims(cat_df: pd.DataFrame) -> dict[str, int]:
    """
    Return a dict { col_name : ℓ_j } where ℓ_j = max(2, floor(m_j/2)).
    Here m_j = number of unique categories in column j.
    """
    dims = {}
    for col in cat_df.columns:
        m_j = cat_df[col].nunique()
        dims[col] = max(2, m_j // 2)
    return dims



def build_softmax_per_feat_autoencoder(one_hot_matrix: np.ndarray,
                                        block_sizes: list[int],
                                        hidden_dims: list[int],
                                        learning_rate: float) -> models.Model:

    n_total = one_hot_matrix.shape[1]
    bottleneck = int(np.sum(hidden_dims))

    x_in = layers.Input(shape=(n_total,), name="AE_input")

    # Encoder: linear, no bias
    encoded = layers.Dense(bottleneck,
                           activation=None,
                           use_bias=False,
                           name="encoder_dense")(x_in)

    # Decoder: linear, with bias
    decoded_lin = layers.Dense(n_total,
                               activation=None,
                               use_bias=True,
                               name="decoder_dense")(encoded)

    # slice into per‐feature blocks and apply a softmax on each block
    soft_blocks = []
    cursor = 0
    for i, m_j in enumerate(block_sizes):
        # Slice out the logits for feature j (size = m_j)
        block = layers.Lambda(lambda t, s=cursor, e=cursor + m_j: t[:, s:e])(decoded_lin)

        # Dense(m_j) with "softmax" activation
        # and initialize weights = [Identity(m_j×m_j), zeros(m_j)].
        head = layers.Dense(m_j,
                            activation="softmax",
                            use_bias=True,
                            trainable=False,
                            name=f"softmax_feat_{i}")
        soft = head(block)

        # Manually set the (kernel, bias) = (I_{m_j}, 0)
        identity_kernel = np.eye(m_j, dtype=np.float32)
        zero_bias = np.zeros((m_j,), dtype=np.float32)
        head.set_weights([identity_kernel, zero_bias])

        soft_blocks.append(soft)
        cursor += m_j

    assert cursor == n_total, "Sum of block_sizes must equal total one‐hot width."

    # Concatenate all per-feature softmax outputs
    out = layers.Concatenate(name="concat_softmax")(soft_blocks)

    autoencoder = models.Model(inputs=x_in, outputs=out, name="joint_autoencoder")
    autoencoder.compile(
        optimizer=optimizers.Nadam(learning_rate=learning_rate),
        loss="categorical_crossentropy"
    )
    return autoencoder

In [ ]:
df = pd.read_csv("dmean_df.csv")

In [ ]:
# storing indices of patients to ensure 80/20 split 
y = df["ExtractionFlag"]
X = df.drop(columns=["ExtractionFlag"])

X_dummies = pd.get_dummies(
    X,
    columns=[c for c in X.columns if c not in ["TotalDose", "PatientID"]],
    drop_first=False,
    dtype=float
)

unique_patients = df['PatientID'].unique()

train_patients, test_patients = train_test_split(
    unique_patients, 
    test_size=0.2, 
    random_state=42
)

train_mask = df['PatientID'].isin(train_patients)
test_mask = df['PatientID'].isin(test_patients)

df = df.drop(columns=["PatientID"])

In [ ]:
target_col = "ExtractionFlag"
cont_cols = ["TotalDose"]
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found in CSV.")
y = df[target_col].astype(int).values           

for c in cont_cols:
    if c not in df.columns:
        raise ValueError(f"Continuous column '{c}' not found in CSV.")
cat_cols = df.drop(columns=[target_col, *cont_cols]).columns.tolist()

print("Target column:    ExtractionFlag")
print("Continuous cols: ", cont_cols)
print("Categorical cols:", cat_cols, "\n")

#  CONTINUOUS FEATURES 
data_cont_scaled = min_max_scaler(df[cont_cols].astype(float).values) 
data_cont_matrix = data_cont_scaled.astype(np.float32)

# CATEGORICAL DATAFRAME 
data_cat = df[cat_cols]
col_names = list(data_cat.columns)

# ONE-HOT ENCODING
one_hot_encoders: dict[str, OneHotEncoder] = {}
one_hot_arrays = []
block_sizes = []

print("Building one‐hot encoding for each feature…")
for col in col_names:
    original_values = data_cat[[col]].astype(str)        
    ohe = OneHotEncoder(sparse_output=False, dtype=np.float32)
    one_hot = ohe.fit_transform(original_values)         # (n_samples, m_j)
    one_hot_encoders[col] = ohe
    m_j = one_hot.shape[1]

    one_hot_arrays.append(one_hot)
    block_sizes.append(m_j)

    print(f"  • {col}: {m_j} OHE features")

# OHE matrix
X_cat_onehot = np.concatenate(one_hot_arrays, axis=1)
print(f"\nConcatenated one‐hot matrix shape: {X_cat_onehot.shape}\n")

# Compute hidden_dims per feature
cat_df_for_dims = data_cat.astype(str)
hidden_dims_dict = per_feature_hidden_dims(cat_df_for_dims)
hidden_dims = [hidden_dims_dict[col] for col in col_names]
print("Per‐feature hidden_dims:", hidden_dims, "\n")

# BUILD JOINT AUTOENCODER
learning_rate = 0.01
batch_size = 64
epochs = 50
validation_split = 0.1

joint_ae = build_softmax_per_feat_autoencoder(
    one_hot_matrix=X_cat_onehot,
    block_sizes=block_sizes,
    hidden_dims=hidden_dims,
    learning_rate=learning_rate
)
joint_ae.summary()

# EarlyStopping callback
es = callbacks.EarlyStopping(monitor="val_loss",
                             patience=5,
                             restore_best_weights=True)

history = joint_ae.fit(
    x=X_cat_onehot,
    y=X_cat_onehot,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=validation_split,
    callbacks=[es],
    verbose=2
)

print("\nJoint autoencoder training complete.\n")

# RECONSTRUCTION ACCURACY PER FEATURE
pred_all = joint_ae.predict(X_cat_onehot, batch_size=512, verbose=0)
cursor = 0
for i, col in enumerate(col_names):
    m_j = block_sizes[i]
    pred_block = pred_all[:, cursor : cursor + m_j]
    true_block = X_cat_onehot[:, cursor : cursor + m_j]

    pred_labels = np.argmax(pred_block, axis=1)
    true_labels = np.argmax(true_block, axis=1)
    acc = (pred_labels == true_labels).mean() * 100

    print(f"  → {col}: rec acc={acc:.1f}%")
    cursor += m_j
print()  

encoder_layer = joint_ae.get_layer("encoder_dense")
encoder_model = models.Model(inputs=joint_ae.input,
                             outputs=encoder_layer.output)

X_cat_embedding = encoder_model.predict(X_cat_onehot,
                                        batch_size=64,
                                        verbose=0)
print(f"Extracted joint embedding shape: {X_cat_embedding.shape}\n")

parts = []
parts.append(data_cont_matrix)
parts.append(X_cat_embedding)      

X_final = np.concatenate(parts, axis=1)
print(f"Final feature matrix shape: {X_final.shape}\n")

In [37]:
train_idx = np.where(np.asarray(train_mask))[0]
test_idx  = np.where(np.asarray(test_mask))[0]

assert len(train_idx) + len(test_idx) == X_final.shape[0] 
assert set(train_idx).isdisjoint(set(test_idx)), "check for overlap between train/test indices."

# Slice by patient-based indices
X_train = X_final[train_idx]
X_test  = X_final[test_idx]

y_arr = np.asarray(y)  
y_train = y_arr[train_idx]
y_test  = y_arr[test_idx]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, shuffle=True, 
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

X_dummies = pd.get_dummies(
    X,
    columns=[c for c in X.columns if c != "TotalDose"],
    drop_first=False,
    dtype=float
)

X_train, X_test, y_train, y_test = train_test_split(
    X_dummies, y, test_size=0.2, random_state=42
)

print("Training data:", X_train.shape)
print("Testing data: ", X_test.shape)

In [ ]:
# parameter grid
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'class_weight': [
        'balanced',
        {0: 1, 1: 2},
        {0: 1, 1: 4}
    ]
}

# Initialize SVC with RBF kernel
svm_rbf = SVC(kernel='rbf', probability=False, random_state=42)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    svm_rbf,
    param_grid,
    cv=5,
    scoring='accuracy',  
    n_jobs=-1,
    verbose=2
)

# Fit model
grid_search.fit(X_train, y_train)
best_svc = grid_search.best_estimator_

# Best params and evaluation
print("\nBest Parameters:", grid_search.best_params_)
print("\nClassification Report (Test Set):")
y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Wrap & fit a sigmoid calibrator with 5-fold CV
calibrator = CalibratedClassifierCV(
    estimator=grid_search.best_estimator_, 
    cv=5,
    method='sigmoid'
)
calibrator.fit(X_train, y_train)

y_proba_cal = calibrator.predict_proba(X_test)[:, 1]

# Compute calibration‐curve data
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)


# Calibration curves
plt.plot(prob_pred_cal, prob_true_cal, marker='s', label='Calibrated')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.show()

# Print Brier & ROC AUC
print("Calibrated   Brier:", brier_score_loss(y_test, y_proba_cal))
print("Calibrated   ROC AUC:", roc_auc_score(y_test, y_proba_cal))

In [ ]:
# Weighted cross-entropy (log-loss with class weights)
classes = np.unique(y_test)
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_test)
class_weight_dict = dict(zip(classes, cw))
sample_weights = np.array([class_weight_dict[y] for y in y_test])
wlog = log_loss(y_test, y_proba_cal, sample_weight=sample_weights)
print(f"Weighted log-loss: {wlog:.4f}")

# Confusion matrix & classification report at threshold=0.5
for thr in [0.5]:
    y_thr = (y_proba_cal >= thr).astype(int)
    cm_thr = confusion_matrix(y_test, y_thr)
    print(f"\nThreshold={thr} Confusion Matrix:\n{cm_thr}")
    print(classification_report(y_test, y_thr))